# First-iteration Zeus runs, rows 9–16: logarithms in ${}_2F_1(z)$

Runs 9–16 of `RemoteRuns/runs_first_iteration.md` (Zeus jobs 4857455–4857463, 2026-11-02, RD commit
4a797a8, Simulated Annealing over **all shards**, walk depth 1000, trajectory length $\|v\|_1 \le 100$).
The spec lists these rows as forward (F) seeds. They were run as dual (D) walks after the pFq
type 2 → type 1 conversion: F ray $(x;y)$, $(dx;dy)$ → D ray $(1-x;\,2-y)$, $(-dx;\,-dy)$ in the same
${}_2F_1(z)$.

This notebook does not use RD. Each best trajectory is **replicated independently**:

1. Walk the ${}_2F_1(z)$ CMF of `ramanujantools` exactly: $W_N = M(0)\,M(1)\cdots M(N-1)$ along the
   trajectory, take $S_N = W_N^{-T}$ (D) or $S_N = W_N$ (F), and normalise its first column:
   $c = (1, c_1)$ with $c_j = (S_N)_{j0}/(S_N)_{00}$.
2. Give LIReC the constant and $c_1$ to 1000 digits; it returns the Möbius form
   $L = (p_0 + p_1 c_1)/(q_0 + q_1 c_1)$. A walk with fewer than 1000 correct digits (only the log 7
   seed here) is retried with half its correct digits and checked on digits LIReC never saw.
3. Evaluate that form **exactly** (rational arithmetic) at depth $N$, reduce it to $P_N/Q_N$, and compute
   $\delta_N = -1 - \log|P_N/Q_N - L| / \log Q_N$, i.e. $|L - P_N/Q_N| = Q_N^{-(1+\delta_N)}$.

δ is computed at N = 1000 (RD's search depth, and the depth that matches the paper's Table 3),
then at N = 2000, 4000 and 8000 (the depth of the paper's Table 2) to see whether it holds up.

Each best trajectory is then **run from scratch** without any CMF library ("From scratch" blocks):
the ${}_2F_1$ matrices are built by hand from the hypergeometric differential equation, the trajectory
matrix $M(n)$ is multiplied out by hand, and the walk is reduced to a **scalar order-2 recurrence with
polynomial coefficients and explicit initial values**. That recurrence alone reproduces the same rationals
and the same δ up to N = 8000.

## Where the pFq conversion was (and was not) used

* **Used once, before the search:** the spec's F seeds were turned into D start points for the Simulated
  Annealing search (e.g. row 9: F (2, 2; 4)/(1, 1; 2) → D (−1, −1; −2)/(−1, −1; −2)). That only chose where the
  search started. The search then moved freely through all shards of the CMF.
* **Not used for any result:** every best trajectory below is an ordinary dual walk (D, $W_N^{-T}$, the
  paper's "D" walks) in the standard ${}_2F_1(z)$ CMF, at the start point and direction printed. No
  conversion is applied when it is evaluated. The from-scratch blocks show it is nothing but a
  polynomial recurrence whose ratio converges to the constant. Whether it came from a D or an F walk
  does not matter for δ.

## Comparison conventions

* **Paper.** arXiv:2610.27799, Table 3 (p. 43): Alladi–Robinson (1980) constructions, $\delta_N$ of
  forward walks. The seeds are recomputed here with the same code, so both sides are measured the same way.
* **Literature record.** A published irrationality-measure bound $\mu(L) \le \mu_0$ corresponds to
  $\delta_{eq} = 1/(\mu_0 - 1)$ (the paper's convention, "Record δ" in its Table 2). A numerical
  $\delta_N$ above $\delta_{eq}$ is **evidence**, not a proof: the proof also needs the arithmetic of the
  denominators (a divisibility lemma) and an asymptotic, not finite-depth, exponent.
* **Not records.** $\delta < 0$ never improves an irrationality measure (every finite $\mu_0$ gives
  $\delta_{eq} > 0$).

In [1]:
import time

import mpmath as mp
import sympy as sp
from LIReC.db.access import db
from ramanujantools import Position
from ramanujantools.cmf import pFq

R = sp.Rational
n = sp.Symbol('n')
DEPTHS = (1000, 2000, 4000, 8000)


def walk_column(z, start, direction, depth, dual):
    """Normalised first column (1, c1) of W^{-T} (dual, D) or W (forward, F), W = M(0)···M(depth-1).

    Coordinates are in ramanujantools' axis order (x0, x1; y0).
    """
    cmf = pFq(2, 1, z)
    keys = list(cmf.matrices.keys())
    m = cmf.trajectory_matrix(Position(dict(zip(keys, direction))), Position(dict(zip(keys, start))))
    w = m.walk({n: 1}, depth, {n: 0})
    s = w.inv().T if dual else w
    col = s.col(0)
    return [R(x) / R(col[0]) for x in col]


def _lirec(C, column, digits):
    """One LIReC call on (C, c1) to `digits` digits -> (relation text, p, q), or (None, None, None)."""
    res = db.identify([C.evalf(digits)] + [x.evalf(digits) for x in column[1:]])
    if not res:
        return None, None, None
    text = str(res[0])
    rhs = sp.sympify(text.split('=', 1)[1].rsplit(' (', 1)[0])
    num, den = sp.fraction(sp.together(rhs))
    c1 = sp.Symbol('c1')
    p = [num.subs(c1, 0), sp.diff(num, c1)]
    q = [den.subs(c1, 0), sp.diff(den, c1)]
    return text, [int(x) for x in p], [int(x) for x in q]


def stable_digits(column, column_09):
    """Correct digits of the walk, estimated from c1's agreement between depth 0.9·N and N."""
    with mp.workdps(60):
        return int(-mp.log10(abs(mp.mpf(sp.N(column[1] - column_09[1], 40)) / mp.mpf(sp.N(column[1], 40)))))


def lirec_form(C, column, column_09):
    """LIReC form C = (p·c)/(q·c) of the walk column -> (relation text, p, q, digits given to LIReC).

    First 1000 digits. LIReC only accepts relations that hold to its input precision, so a walk with fewer
    correct digits s is retried with s/2 digits, and the form is accepted only if it reproduces C to
    0.9·s digits, digits LIReC never saw (RD's rule since PR #11).
    """
    text, p, q = _lirec(C, column, 1000)
    if text is not None:
        return text, p, q, 1000
    s = stable_digits(column, column_09)
    text, p, q = _lirec(C, column, s // 2)
    if text is None or score(C, approximant(column, p, q))[1] < 0.9 * s:
        return None, None, None, s // 2
    return text, p, q, s // 2


def approximant(column, p, q):
    """Exact reduced rational P/Q = (p·c)/(q·c)."""
    return R(sum(a * x for a, x in zip(p, column)), sum(b * x for b, x in zip(q, column)))


def score(C, r):
    """(δ, correct digits, log10 Q) of the reduced rational r = P/Q against C."""
    lq = len(str(abs(r.q)))
    with mp.workdps(2 * lq + 100):
        err = abs(mp.mpf(r.p) / r.q - mp.mpf(sp.N(C, mp.mp.dps)))
        return float(-1 - mp.log(err) / mp.log(abs(r.q))), float(-mp.log10(err)), float(mp.log10(abs(r.q)))

## The data

`best` is the best trajectory of each Zeus run (from the run's `search results/*.jsonl`; start, direction
and RD's δ at depth 1000). `seed` is the paper's Alladi–Robinson construction (Table 3, forward walk).
`lit` is the best published bound found in a literature search (2026-11-02); see the notes below the
summary table.

In [2]:
RUNS = {   # constant: (value, z, best D start, best D direction, RD δ@1000, paper F seed start, direction, paper δ)
    'log(3/2)': (sp.log(R(3, 2)), R(-1, 2), (1, 1, 3), (14, 13, 26), 0.4315018067463825, (2, 2, 4), (1, 1, 2), 0.408),
    'log(4/3)': (sp.log(R(4, 3)), R(-1, 3), (1, 1, 3), (19, 18, 36), 0.4754538555025647, (2, 2, 4), (1, 1, 2), 0.461),
    'log(5/4)': (sp.log(R(5, 4)), R(-1, 4), (1, 1, 3), (20, 21, 42), 0.5048228274827125, (1, 1, 2), (1, 1, 2), 0.497),
    'log(6/5)': (sp.log(R(6, 5)), R(-1, 5), (1, 1, 3), (23, 22, 44), 0.5260130735769053, (1, 1, 2), (1, 1, 2), 0.518),
    'log(5/3)': (sp.log(R(5, 3)), R(-2, 3), (1, 1, 3), (19, 18, 36), 0.11564693193213316, (1, 1, 2), (1, 1, 2), 0.106),
    'log(7/5)': (sp.log(R(7, 5)), R(-2, 5), (0, 0, 0), (-21, -22, -44), 0.19871284530381672, (1, 1, 2), (1, 1, 2), 0.194),
    'log 5': (sp.log(5), -4, (1, 1, 3), (13, 14, 26), -0.4094849618813896, (1, 1, 2), (1, 1, 2), -0.420),
    'log 7': (sp.log(7), -6, (0, 0, 0), (-18, -19, -36), -0.5472966441084282, (1, 1, 2), (1, 1, 2), -0.553),
}

# Best published irrationality-measure bound found: (mu_0, source). delta_eq = 1/(mu_0 - 1).
# Alladi–Robinson: explicit bound for log(1 + b/a) (Zeilberger–Zudilin's formulation, computed below).
def alladi_robinson_mu(a, b):
    """Alladi–Robinson bound for mu(log(1 + b/a)); None when the construction gives no measure."""
    bm, bp = 2 * a + b - 2 * mp.sqrt(a * (a + b)), 2 * a + b + 2 * mp.sqrt(a * (a + b))
    den = mp.log(bm) + 1
    return None if den >= 0 else float((mp.log(bm) - mp.log(bp)) / den)

LIT = {
    'log(3/2)': (alladi_robinson_mu(2, 1), 'Alladi–Robinson 1980 (Rhin 1987: 8.616 for Q·log2+Q·log3 is weaker)'),
    'log(4/3)': (alladi_robinson_mu(3, 1), 'Alladi–Robinson 1980 (Rhin 1987: 8.616 is weaker)'),
    'log(5/4)': (alladi_robinson_mu(4, 1), 'Alladi–Robinson 1980'),
    'log(6/5)': (alladi_robinson_mu(5, 1), 'Alladi–Robinson 1980'),
    'log(5/3)': (5.5119417, 'Salikhov–Zolotukhina–Tomashevskaya 2019 (Chebyshevskii Sb. 20(4))'),
    'log(7/5)': (4.865, 'cited by Luchin (Chebyshevskii Sb.), Salikhov school'),
    'log 5': (None, 'finite bounds exist (Wu 2002/03, Baker-type); any finite mu_0 gives delta_eq > 0'),
    'log 7': (None, 'Wu 2002; Bondareva–Luchin–Salikhov 2020 (Math. Notes 107); delta_eq > 0'),
}
for k, (mu0, src) in LIT.items():
    print(f'{k:9s} mu_0 = {mu0 if mu0 is None else round(mu0, 6)!s:10s} delta_eq = '
          f'{"> 0" if mu0 is None else round(1 / (mu0 - 1), 4)!s:7s} {src}')

log(3/2)  mu_0 = 3.547471   delta_eq = 0.3925  Alladi–Robinson 1980 (Rhin 1987: 8.616 for Q·log2+Q·log3 is weaker)
log(4/3)  mu_0 = 3.224053   delta_eq = 0.4496  Alladi–Robinson 1980 (Rhin 1987: 8.616 is weaker)
log(5/4)  mu_0 = 3.059731   delta_eq = 0.4855  Alladi–Robinson 1980
log(6/5)  mu_0 = 2.95741    delta_eq = 0.5109  Alladi–Robinson 1980
log(5/3)  mu_0 = 5.511942   delta_eq = 0.2216  Salikhov–Zolotukhina–Tomashevskaya 2019 (Chebyshevskii Sb. 20(4))
log(7/5)  mu_0 = 4.865      delta_eq = 0.2587  cited by Luchin (Chebyshevskii Sb.), Salikhov school
log 5     mu_0 = None       delta_eq = > 0     finite bounds exist (Wu 2002/03, Baker-type); any finite mu_0 gives delta_eq > 0
log 7     mu_0 = None       delta_eq = > 0     Wu 2002; Bondareva–Luchin–Salikhov 2020 (Math. Notes 107); delta_eq > 0


In [3]:
def replicate(name, depths=DEPTHS):
    """Walk the best (D) trajectory and the paper seed (F); LIReC at N=1000; exact δ at every depth.

    The form (p, q) is identified once at N = 1000 and kept fixed; at N = 2000 LIReC is run again on the
    deeper column as a check that it returns the same form.
    """
    C, z, s, d, rd_delta, s_f, d_f, paper = RUNS[name]
    out = {'name': name, 'rd_delta': rd_delta, 'paper': paper}
    for tag, (st, dr, dual) in (('best', (s, d, True)), ('seed', (s_f, d_f, False))):
        col = walk_column(z, st, dr, 1000, dual)
        text, p, q, digits = lirec_form(C, col, walk_column(z, st, dr, 900, dual))
        rows, same_form_2000 = [], None
        for N in depths:
            t = time.time()
            col_n = col if N == 1000 else walk_column(z, st, dr, N, dual)
            if N == 2000:
                _, p2, q2, _ = lirec_form(C, col_n, walk_column(z, st, dr, 1800, dual))
                same_form_2000 = (p2, q2) in ((p, q), ([-x for x in p], [-x for x in q]))
            dlt, dig, lq = score(C, approximant(col_n, p, q))
            rows.append((N, dlt, dig, lq, time.time() - t))
        out[tag] = dict(start=st, direction=dr, walk='D' if dual else 'F', lirec=text, p=p, q=q, rows=rows,
                        same_form_2000=same_form_2000, lirec_digits=digits)
    return out


def show(name):
    """Print one constant's replication (best trajectory, then the paper's seed)."""
    res = RESULTS[name]
    for tag in ('best', 'seed'):
        b = res[tag]
        print(f'{name} — {tag}: {b["walk"]} walk, start {b["start"]}, direction {b["direction"]}')
        print(f'   LIReC (N=1000, {b["lirec_digits"]} digits): {b["lirec"]}  ->  p = {b["p"]}, q = {b["q"]};'
              f'  same form from LIReC at N=2000: {b["same_form_2000"]}')
        for N, dlt, dig, lq, sec in b['rows']:
            print(f'   N={N:5d}  δ_N = {dlt:+.6f}   correct digits {dig:9.0f}   log10 Q {lq:9.0f}   ({sec:.1f} s)')
    b1000 = res['best']['rows'][0][1]
    print(f'   RD (Zeus) δ@1000 = {res["rd_delta"]:+.13f}; independent δ@1000 = {b1000:+.13f}; '
          f'|diff| = {abs(b1000 - res["rd_delta"]):.1e}.  Paper Table 3: {res["paper"]:+.3f}')


RESULTS = {}

## From scratch: building and running a trajectory without any CMF library

**Matrices.** ${}_2F_1(x_0, x_1; y_0; z)$ satisfies $\theta(\theta + y_0 - 1)F = z(\theta + x_0)(\theta + x_1)F$
with $\theta = z\,d/dz$. Let $\Theta$ be the companion matrix of the monic $\theta$-polynomial. The contiguous
relations give $M_{x_i} = \Theta/x_i + I$ (raise $x_i$ by one) and $\Theta/(y_0-1) + I$ (lower $y_0$ by
one). The matrix that raises $y_0$ is therefore $M_{y_0} = \big(\Theta|_{y_0\to y_0+1}/y_0 + I\big)^{-1}$.
Below these are checked to equal `ramanujantools`' matrices, and to commute: $M_a(p)M_b(p+e_a) = M_b(p)M_a(p+e_b)$.

**Trajectory matrix.** $M(n)$ is the product of the unit-step matrices along the lattice path from
$s + n v$ to $s + (n+1) v$ (all $x_0$ steps, then $x_1$, then $y_0$; a negative step uses
$M_a(p - e_a)^{-1}$). It is multiplied out **symbolically in $n$**, so poles and zeros of single steps
cancel. Evaluating step by step at lattice points can hit a singular unit matrix (e.g. $\det M_{x_0} = 0$
at $x_0 = y_0 - 1$), although $M(n)$ itself is regular.

**Walk.** $W_N = M(0)M(1)\cdots M(N-1)$ and $S_N = W_N^{-T}$, so $(S_N)_{j0} = (W_N^{-1})_{0j}$. The columns of
$W_N^{-1}$ obey $v_N = M(N-1)^{-1} v_{N-1}$ with $v_0 = e_j$. The *plain loop* below multiplies integer
matrices $\propto \operatorname{adj} M(k)$ (scalars cancel in the ratio).

**Scalar recurrence.** First components of solutions of a 2×2 first-order system satisfy one order-2
recurrence $D(n)\,a_{n+2} = E(n)\,a_{n+1} + F(n)\,a_n$, derived here from $M(n)$ by elimination. Numerator
$A_N = p\cdot(x_N, y_N)$ and denominator $B_N = q\cdot(x_N, y_N)$ (LIReC's form) satisfy it with initial values
$A_0 = p_0$, $A_1 = p\cdot(\text{row 0 of } M(0)^{-1})$, and the same for $B$. Then $P_N/Q_N = A_N/B_N$.

In [4]:
import gmpy2
from gmpy2 import mpz

x0, x1, y0, th = sp.symbols('x0 x1 y0 theta')
AX = (x0, x1, y0)


def cmf_matrices(z):
    """The 2F1(z) CMF built by hand from θ(θ + y0 − 1) − z(θ + x0)(θ + x1) = 0."""
    a2, a1, a0 = sp.Poly(sp.expand(th * (th + y0 - 1) - z * (th + x0) * (th + x1)), th).all_coeffs()
    Theta = sp.Matrix([[0, -a0 / a2], [1, -a1 / a2]])
    I2 = sp.eye(2)
    lower_y = Theta / (y0 - 1) + I2
    return {x0: Theta / x0 + I2, x1: Theta / x1 + I2, y0: lower_y.subs(y0, y0 + 1).inv().applyfunc(sp.cancel)}


def check_matrices(z, mats):
    """(equal to ramanujantools' matrices, flat) for the hand-made matrices."""
    rt = pFq(2, 1, z).matrices
    same = all((mats[a] - sp.Matrix(rt[a])).applyfunc(sp.cancel) == sp.zeros(2) for a in AX)
    flat = all((mats[a] * mats[b].subs(a, a + 1) - mats[b] * mats[a].subs(b, b + 1)).applyfunc(sp.cancel)
               == sp.zeros(2) for a in AX for b in AX)
    return same, flat


def trajectory_matrix(mats, start, direction):
    """M(n): product of unit steps from start + n·direction to start + (n+1)·direction, symbolic in n."""
    pos = [s + n * d for s, d in zip(start, direction)]
    m = sp.eye(2)
    for i, ax in enumerate(AX):
        for _ in range(abs(direction[i])):
            sub = dict(zip(AX, pos))
            if direction[i] > 0:
                step = mats[ax].subs(sub)
                pos[i] += 1
            else:
                sub[ax] -= 1
                step = mats[ax].subs(sub).inv()
                pos[i] -= 1
            m = (m * step).applyfunc(sp.cancel)
    return m


def integer_poly_matrix(mat):
    """Callables k -> integer 2x2 matrix proportional to mat(k) (common denominator dropped)."""
    den = sp.lcm([sp.fraction(e)[1] for e in mat])
    polys = [sp.Poly(sp.cancel(e * den), n) for e in mat]
    scale = sp.ilcm(*[sp.fraction(c)[1] for p in polys for c in p.all_coeffs()])
    fns = [sp.lambdify(n, (p * scale).as_expr(), 'math') for p in polys]
    return lambda k: [[int(fns[0](k)), int(fns[1](k))], [int(fns[2](k)), int(fns[3](k))]]


REDUCE_EVERY = 16   # divide the integer state by its common gcd this often (keeps numbers near Q's size)


def _reduce(values):
    """Divide a list of GMP integers by their common gcd (the ratio is all that matters)."""
    g = 0
    for v in values:
        g = gmpy2.gcd(g, v)
    return [v // g for v in values] if g > 1 else values


def plain_loop(m, p, q, depths):
    """D walk by plain integer matrix products: X_N ∝ W_N^{-1} = adj M(N-1) ··· adj M(0); returns {N: P_N/Q_N}."""
    adj = integer_poly_matrix(m.adjugate().applyfunc(sp.cancel))
    X, out = [mpz(1), mpz(0), mpz(0), mpz(1)], {}
    for k in range(max(depths)):
        u = [mpz(v) for row in adj(k) for v in row]
        X = [u[0] * X[0] + u[1] * X[2], u[0] * X[1] + u[1] * X[3],
             u[2] * X[0] + u[3] * X[2], u[2] * X[1] + u[3] * X[3]]
        if k % REDUCE_EVERY == 0:
            X = _reduce(X)
        if k + 1 in depths:
            out[k + 1] = R(int(p[0] * X[0] + p[1] * X[1]), int(q[0] * X[0] + q[1] * X[1]))
    return out


def scalar_recurrence(m):
    """Integer polynomials D, E, F with D(n) a_{n+2} = E(n) a_{n+1} + F(n) a_n for first components of
    v_N = M(N-1)^{-1} v_{N-1}, and T0 = M(0)^{-1}."""
    minv = m.inv().applyfunc(sp.cancel)
    T = lambda k: minv.subs(n, k - 1)
    t2, t1 = T(n + 2), T(n + 1)
    assert sp.cancel(t1[0, 1]) != 0
    c1 = sp.cancel(t2[0, 0] + t2[0, 1] * t1[1, 1] / t1[0, 1])
    c0 = sp.cancel(t2[0, 1] * (t1[1, 0] - t1[1, 1] * t1[0, 0] / t1[0, 1]))
    den = sp.lcm(sp.fraction(c1)[1], sp.fraction(c0)[1])
    D, E, F = (sp.Poly(sp.cancel(e), n) for e in (den, c1 * den, c0 * den))
    g = sp.gcd(sp.gcd(D, E), F)
    D, E, F = (x.quo(g) for x in (D, E, F))
    scale = sp.ilcm(*[sp.fraction(c)[1] for x in (D, E, F) for c in x.all_coeffs()])
    return [x * scale for x in (D, E, F)], minv.subs(n, 0)


def run_recurrence(DEF, T0, p, q, depths):
    """A_N / B_N from the scalar recurrence alone (integer state, common scale per step); {N: P_N/Q_N}."""
    fD, fE, fF = (sp.lambdify(n, x.as_expr(), 'math') for x in DEF)
    A0, A1 = R(p[0]), p[0] * T0[0, 0] + p[1] * T0[0, 1]
    B0, B1 = R(q[0]), q[0] * T0[0, 0] + q[1] * T0[0, 1]
    l = sp.ilcm(A0.q, A1.q, B0.q, B1.q)
    a0, a1, b0, b1 = (mpz(int(v * l)) for v in (A0, A1, B0, B1))
    out = {1: R(int(a1), int(b1))} if 1 in depths else {}
    for k in range(max(depths) - 1):
        d, e, f = mpz(fD(k)), mpz(fE(k)), mpz(fF(k))
        a1, b1, a0, b0 = e * a1 + f * a0, e * b1 + f * b0, d * a1, d * b1
        if k % REDUCE_EVERY == 0:
            a1, b1, a0, b0 = _reduce([a1, b1, a0, b0])
        if k + 2 in depths:
            out[k + 2] = R(int(a1), int(b1))
    return out, (A0, A1, B0, B1)


def rt_approximant(z, start, direction, depth, p, q):
    """The same approximant through ramanujantools' walk (for the equality check)."""
    col = walk_column(z, start, direction, depth, dual=True)
    return approximant(col, p, q)


def from_scratch(name, loop_depths=(1000, 2000), rec_depths=DEPTHS, show_recurrence=True):
    """Rebuild the best trajectory of `name` by hand and run it start to end."""
    C, z, s, d = RUNS[name][:4]
    p, q = RESULTS[name]['best']['p'], RESULTS[name]['best']['q']
    t = time.time()
    mats = cmf_matrices(z)
    same, flat = check_matrices(z, mats)
    m = trajectory_matrix(mats, s, d)
    rt = pFq(2, 1, z)
    keys = list(rt.matrices.keys())
    m_rt = sp.Matrix(rt.trajectory_matrix(Position(dict(zip(keys, d))), Position(dict(zip(keys, s)))))
    same_m = (m - m_rt).applyfunc(sp.cancel) == sp.zeros(2)
    DEF, T0 = scalar_recurrence(m)
    c1 = sp.Symbol('c1')
    print(f'From scratch — {name}: 2F1({z}), D walk, start (x0, x1; y0) = {s}, direction {d}; '
          f'form C = {(p[0] + p[1] * c1) / (q[0] + q[1] * c1)}')
    print(f'   hand-made matrices = ramanujantools: {same}; flat: {flat}; '
          f'hand-made M(n) = ramanujantools trajectory matrix: {same_m}')
    print(f'   det M(n) = {sp.factor(m.det())}')
    print(f'   scalar recurrence D(n) a(n+2) = E(n) a(n+1) + F(n) a(n): deg D, E, F = '
          f'{DEF[0].degree()}, {DEF[1].degree()}, {DEF[2].degree()}')
    if show_recurrence:
        print(f'   D(n) = {sp.factor(DEF[0].as_expr())}')
        print(f'   E(n) = {DEF[1].as_expr()}')
        print(f'   F(n) = {sp.factor(DEF[2].as_expr())}')
    rec, (A0, A1, B0, B1) = run_recurrence(DEF, T0, p, q, set(rec_depths) | {1, 2, 3, 4})
    print(f'   initial values: A0 = {A0}, A1 = {A1};  B0 = {B0}, B1 = {B1}')
    for N in (1, 2, 3, 4):
        r = rec[N]
        print(f'   N={N}: A/B = {sp.N(r, 25)}  (|A/B − C| = 10^−{score(C, r)[1]:.1f}, Q has {len(str(r.q))} digits)')
    loop = plain_loop(m, p, q, set(loop_depths))
    print(f'   {"N":>5s}  {"δ_N (recurrence)":>17s}  {"= plain loop":>12s}  {"= ramanujantools":>16s}')
    for N in rec_depths:
        r = rec[N]
        eq_loop = (r == loop[N]) if N in loop else '—'
        eq_rt = (r == rt_approximant(z, s, d, N, p, q)) if N in loop_depths else '—'
        print(f'   {N:5d}  {score(C, r)[0]:+17.6f}  {eq_loop!s:>12s}  {eq_rt!s:>16s}')
    print(f'   ({time.time() - t:.0f} s)')

## Exact replication of one walk, step by step (log(6/5))

The approximant at depth 1000 is an exact rational. Its first and last digits are printed so the walk can
be checked against any other implementation.

In [5]:
C, z, s, d = RUNS['log(6/5)'][:4]
col = walk_column(z, s, d, 1000, dual=True)
text, p, q, digits = lirec_form(C, col, walk_column(z, s, d, 900, dual=True))
print(f'LIReC relation ({digits} digits):', text)
r = approximant(col, p, q)
P, Q = str(r.p), str(r.q)
print(f'P_1000 = {P[:30]}…{P[-30:]}  ({len(P)} digits)')
print(f'Q_1000 = {Q[:30]}…{Q[-30:]}  ({len(Q)} digits)')
print(f'P/Q    = {sp.N(r, 60)}')
print(f'log 6/5= {sp.N(C, 60)}')
print('δ_1000 =', score(C, r)[0])

LIReC relation (1000 digits): c0 = (-c1 - 2)/(-6*c1 - 11) (1000)
P_1000 = 814974418689627478157502853931…192188154517010500204371413899  (38669 digits)
Q_1000 = 446998387366035370262575710318…000000000000000000000000000000  (38670 digits)
P/Q    = 0.182321556793954626211718025154514633197389337914486983942726
log 6/5= 0.182321556793954626211718025154514633197389337914486983942726


δ_1000 = 0.5260130735750121


## Summary

Each best trajectory compared with the paper's construction (recomputed) and with the best literature
bound found. The verdict uses δ at the deepest depth computed (N = 8000).

In [6]:
for name in RUNS:
    RESULTS[name] = replicate(name)

print(f'{"constant":9s} {"paper T3":>8s} {"seed@1000":>9s} {"best@1000":>9s} {"best@2000":>9s} {"best@8000":>9s} '
      f'{"seed@8000":>9s} {"δ_eq lit":>8s}  verdict')
CANDIDATES = []
for name, res in RESULTS.items():
    b = {N: dlt for N, dlt, *_ in res['best']['rows']}
    sd = {N: dlt for N, dlt, *_ in res['seed']['rows']}
    mu0 = LIT[name][0]
    deq = None if mu0 is None else 1 / (mu0 - 1)
    beats_paper = b[1000] > sd[1000] and b[8000] > sd[8000]
    if b[8000] <= 0:
        verdict = 'better than paper; δ < 0, no measure'
    elif deq is not None and b[8000] > deq:
        verdict = 'WORLD-RECORD CANDIDATE (numerical)'
        CANDIDATES.append(name)
    else:
        verdict = 'better than paper; below literature' if beats_paper else 'below literature'
    print(f'{name:9s} {res["paper"]:+8.3f} {sd[1000]:+9.4f} {b[1000]:+9.4f} {b[2000]:+9.4f} {b[8000]:+9.4f} '
          f'{sd[8000]:+9.4f} {"   > 0" if deq is None else f"{deq:8.4f}"}  {verdict}')
print('\nworld-record candidates:', CANDIDATES)

constant  paper T3 seed@1000 best@1000 best@2000 best@8000 seed@8000 δ_eq lit  verdict
log(3/2)    +0.408   +0.4034   +0.4315   +0.4292   +0.4293   +0.3954   0.3925  WORLD-RECORD CANDIDATE (numerical)
log(4/3)    +0.461   +0.4608   +0.4755   +0.4746   +0.4732   +0.4511   0.4496  WORLD-RECORD CANDIDATE (numerical)
log(5/4)    +0.497   +0.4972   +0.5048   +0.5034   +0.5039   +0.4878   0.4855  WORLD-RECORD CANDIDATE (numerical)
log(6/5)    +0.518   +0.5176   +0.5260   +0.5252   +0.5249   +0.5128   0.5109  WORLD-RECORD CANDIDATE (numerical)
log(5/3)    +0.106   +0.1056   +0.1156   +0.1148   +0.1138   +0.1000   0.2216  better than paper; below literature
log(7/5)    +0.194   +0.1943   +0.1987   +0.1981   +0.1978   +0.1896   0.2587  better than paper; below literature
log 5       -0.420   -0.4201   -0.4095   -0.4105   -0.4106   -0.4241    > 0  better than paper; δ < 0, no measure
log 7       -0.553   -0.5531   -0.5473   -0.5479   -0.5482   -0.5558    > 0  better than paper; δ < 0, no measure

**Literature notes** (search of 2026-11-02; the sources behind the paywall were not read):

* **log(1 + 1/k), k = 2…5.** The strongest bound found is the explicit Alladi–Robinson bound (1980;
  formulation of Zeilberger–Zudilin, *Automatic discovery of irrationality proofs and irrationality
  measures*, IJNT 2021): μ ≤ 3.5475, 3.2241, 3.0597, 2.9574. Rhin (1987) gives μ < 8.616 for every
  nonzero element of ℚ log 2 + ℚ log 3 (weaker). No later improvement was found for these four values.
  For log 2, Rukhadze (1987) and Marcovecchio (2009) improved Alladi–Robinson with Rukhadze-type
  directions such as (7, 6; 14). Analogous work for k ≥ 2 may exist in sources not read here:
  Heimonen–Matala-aho–Väänänen (Manuscripta Math. 81, 1993), Rhin (1987), Q. Wu (Math. Comp. 72, 2003).
  **These have to be checked before anything is called a record.**
* **log(5/3)**: μ ≤ 5.5119417, Salikhov–Zolotukhina–Tomashevskaya, Chebyshevskii Sbornik 20(4), 2019.
* **log(7/5)**: μ ≤ 4.865, as cited in the Chebyshevskii Sbornik (Luchin, Salikhov school).
* **log 7**: Wu (2002), improved by Bondareva–Luchin–Salikhov, Math. Notes 107 (2020). **log 5**: Wu (2003).
* A self-published, unrefereed PDF by "Dan Brown" (2026-10-07) claims μ(log α) = 2 for all rational α.
  It is not on arXiv or in a journal, and its byline footnote reads "Author of The Da Vinci Code". It is
  ignored here.

## World-record candidates

The trajectories whose numerical δ stays above the best published bound found, at every depth up to 8000.
Each constant has three blocks:
1. the replicated best trajectory (LIReC form, δ_N at N = 1000…8000);
2. the paper's seed, measured the same way;
3. **from scratch**: the trajectory rebuilt by hand and run start to end, with its explicit scalar
   recurrence and initial values. δ_N comes from the recurrence alone and is checked against the plain
   integer loop and `ramanujantools` (N = 1000, 2000).

In [7]:
for name in CANDIDATES:
    show(name)
    print()
    from_scratch(name)
    print()

log(3/2) — best: D walk, start (1, 1, 3), direction (14, 13, 26)
   LIReC (N=1000, 1000 digits): c0 = (-c1 - 2)/(-3*c1 - 5) (999)  ->  p = [-2, -1], q = [-5, -3];  same form from LIReC at N=2000: True
   N= 1000  δ_N = +0.431502   correct digits     25853   log10 Q     18060   (0.1 s)
   N= 2000  δ_N = +0.429246   correct digits     51705   log10 Q     36176   (0.5 s)
   N= 4000  δ_N = +0.428932   correct digits    103407   log10 Q     72367   (1.0 s)
   N= 8000  δ_N = +0.429252   correct digits    206812   log10 Q    144699   (2.4 s)
log(3/2) — seed: F walk, start (2, 2, 4), direction (1, 1, 2)
   LIReC (N=1000, 1000 digits): c0 = (17*c1 - 6)/(42*c1 - 15) (998)  ->  p = [-6, 17], q = [-15, 42];  same form from LIReC at N=2000: True
   N= 1000  δ_N = +0.403392   correct digits      1993   log10 Q      1420   (0.0 s)
   N= 2000  δ_N = +0.399427   correct digits      3985   log10 Q      2847   (0.1 s)
   N= 4000  δ_N = +0.395300   correct digits      7967   log10 Q      5710   (0.0 s)
  

From scratch — log(3/2): 2F1(-1/2), D walk, start (x0, x1; y0) = (1, 1, 3), direction (14, 13, 26); form C = (-c1 - 2)/(-3*c1 - 5)
   hand-made matrices = ramanujantools: True; flat: True; hand-made M(n) = ramanujantools trajectory matrix: True


   det M(n) = -23227183136768*(13*n + 14)*(26*n + 3)**2*(26*n + 5)**2*(26*n + 7)**2*(26*n + 9)**2*(26*n + 11)**2*(26*n + 15)**2*(26*n + 17)**2*(26*n + 19)**2*(26*n + 21)**2*(26*n + 23)**2*(26*n + 25)**2*(26*n + 27)**2/(11907*(n + 1)**2*(3*n + 1)*(3*n + 2)*(4*n + 1)*(4*n + 3)*(6*n + 1)*(6*n + 5)*(7*n + 1)*(7*n + 2)*(7*n + 3)*(7*n + 4)*(7*n + 5)*(7*n + 6)*(12*n + 5)*(12*n + 7)*(12*n + 11)*(12*n + 13)*(13*n + 1)*(14*n + 1)*(14*n + 3)*(14*n + 5)*(14*n + 9)*(14*n + 11)*(14*n + 13))
   scalar recurrence D(n) a(n+2) = E(n) a(n+1) + F(n) a(n): deg D, E, F = 35, 35, 35
   D(n) = 23227183136768*(26*n + 3)*(26*n + 5)*(26*n + 7)*(26*n + 9)*(26*n + 11)*(26*n + 15)*(26*n + 17)*(26*n + 19)*(26*n + 21)*(26*n + 23)*(26*n + 25)*(26*n + 27)*(26*n + 29)*(26*n + 31)*(26*n + 33)*(26*n + 35)*(26*n + 37)*(26*n + 41)*(26*n + 43)*(26*n + 45)*(26*n + 47)*(26*n + 49)*(26*n + 51)*(26*n + 53)*(13280259636445202944576*n**11 + 91429857506914213321408*n**10 + 280594914672965672653968*n**9 + 505999631789651130717936*n*

   initial values: A0 = -2, A1 = -3236918901380873/82240955596800;  B0 = -5, B1 = -70891102621/730300416
   N=1: A/B = 0.4054651081081643819780131  (|A/B − C| = 10^−28.1, Q has 17 digits)
   N=2: A/B = 0.4054651081081643819780131  (|A/B − C| = 10^−53.9, Q has 36 digits)
   N=3: A/B = 0.4054651081081643819780131  (|A/B − C| = 10^−79.7, Q has 51 digits)
   N=4: A/B = 0.4054651081081643819780131  (|A/B − C| = 10^−105.6, Q has 70 digits)


       N   δ_N (recurrence)  = plain loop  = ramanujantools
    1000          +0.431502          True              True


    2000          +0.429246          True              True


... (21 outputs left out)

    1000          +0.526013          True              True


    2000          +0.525204          True              True


    4000          +0.524910             —                 —


    8000          +0.524882             —                 —
   (38 s)



## The other runs

Better than the paper's Alladi–Robinson construction, but below the published bound (log(5/3),
log(7/5)), or with δ < 0 (log 5, log 7). The same three blocks (recurrence polynomials not printed).

In [8]:
for name in RUNS:
    if name not in CANDIDATES:
        show(name)
        print()
        from_scratch(name, show_recurrence=False)
        print()

log(5/3) — best: D walk, start (1, 1, 3), direction (19, 18, 36)
   LIReC (N=1000, 1000 digits): c0 = (2*c1 + 4)/(5*c1 + 8) (999)  ->  p = [4, 2], q = [8, 5];  same form from LIReC at N=2000: True
   N= 1000  δ_N = +0.115647   correct digits     32238   log10 Q     28896   (0.2 s)
   N= 2000  δ_N = +0.114762   correct digits     64474   log10 Q     57837   (0.9 s)
   N= 4000  δ_N = +0.113687   correct digits    128946   log10 Q    115783   (1.6 s)
   N= 8000  δ_N = +0.113816   correct digits    257891   log10 Q    231538   (4.0 s)
log(5/3) — seed: F walk, start (1, 1, 2), direction (1, 1, 2)
   LIReC (N=1000, 1000 digits): c0 = -2*c1/(5 - 5*c1) (999)  ->  p = [0, -2], q = [5, -5];  same form from LIReC at N=2000: True
   N= 1000  δ_N = +0.105641   correct digits      1792   log10 Q      1621   (0.0 s)
   N= 2000  δ_N = +0.104037   correct digits      3585   log10 Q      3247   (0.1 s)
   N= 4000  δ_N = +0.100869   correct digits      7169   log10 Q      6512   (0.0 s)
   N= 8000  δ_N =

From scratch — log(5/3): 2F1(-2/3), D walk, start (x0, x1; y0) = (1, 1, 3), direction (19, 18, 36); form C = (2*c1 + 4)/(5*c1 + 8)
   hand-made matrices = ramanujantools: True; flat: True; hand-made M(n) = ramanujantools trajectory matrix: True


   det M(n) = -19383245667680019896796723*(4*n + 1)**2*(4*n + 3)**2*(12*n + 1)**2*(12*n + 5)**2*(12*n + 7)**2*(12*n + 11)**2*(18*n + 19)*(36*n + 5)**2*(36*n + 7)**2*(36*n + 11)**2*(36*n + 13)**2*(36*n + 17)**2*(36*n + 19)**2*(36*n + 23)**2*(36*n + 25)**2*(36*n + 29)**2*(36*n + 31)**2*(36*n + 35)**2*(36*n + 37)**2/(1615*(n + 1)**2*(17*n + 2)*(17*n + 3)*(17*n + 4)*(17*n + 5)*(17*n + 6)*(17*n + 7)*(17*n + 8)*(17*n + 9)*(17*n + 10)*(17*n + 11)*(17*n + 12)*(17*n + 13)*(17*n + 14)*(17*n + 15)*(17*n + 16)*(17*n + 18)*(18*n + 1)*(19*n + 1)*(19*n + 2)*(19*n + 3)*(19*n + 4)*(19*n + 5)*(19*n + 6)*(19*n + 7)*(19*n + 8)*(19*n + 9)*(19*n + 10)*(19*n + 11)*(19*n + 12)*(19*n + 13)*(19*n + 14)*(19*n + 15)*(19*n + 16)*(19*n + 17)*(19*n + 18))
   scalar recurrence D(n) a(n+2) = E(n) a(n+1) + F(n) a(n): deg D, E, F = 53, 53, 53


   initial values: A0 = 4, A1 = 1154905433959853485057269392/1660281313226519515506975;  B0 = 8, B1 = 2519846950076899642/1850470187193243
   N=1: A/B = 0.5108256237659906832055141  (|A/B − C| = 10^−34.1, Q has 28 digits)
   N=2: A/B = 0.5108256237659906832055141  (|A/B − C| = 10^−66.4, Q has 53 digits)
   N=3: A/B = 0.5108256237659906832055141  (|A/B − C| = 10^−98.6, Q has 82 digits)
   N=4: A/B = 0.5108256237659906832055141  (|A/B − C| = 10^−130.8, Q has 110 digits)


       N   δ_N (recurrence)  = plain loop  = ramanujantools


    1000          +0.115647          True              True


... (20 outputs left out)

       N   δ_N (recurrence)  = plain loop  = ramanujantools
    1000          -0.547297          True              True


    2000          -0.547901          True              True


    4000          -0.548313             —                 —


    8000          -0.548244             —                 —
   (20 s)



## Notes and caveats

* **Replication.** At N = 1000, every δ computed here independently (ramanujantools + LIReC, no RD)
  matches RD's δ from the Zeus runs (difference printed per trajectory). The from-scratch blocks give
  the *identical* rationals (exact equality) through three independent routes: hand-built matrices
  with a plain integer loop, the scalar recurrence, and `ramanujantools`.
* **Integers.** The recurrence has polynomial coefficients and rational initial values. δ uses the
  reduced denominator of $A_N/B_N$, the paper's convention. A proof would need explicit integer
  normalisations (common denominators d_N), which this notebook does not provide.
* **Depth.** From N = 1000 to 8000 the best trajectories lose at most 0.003 (e.g. log(3/2): 0.4315 →
  0.4293, flat from N = 2000 on). The seeds lose 0.005–0.010 (log(3/2): 0.4034 → 0.3954). So the gap
  grows with depth. The values are finite-depth exponents, not limits.
* **Fair comparison.** A best direction such as (23, 22, 44) is about 22 times longer than the seed's
  (1, 1, 2), so at equal N its denominators are much larger (log₁₀ Q column). Even the seed at N = 8000
  has a smaller Q than the best trajectory at N = 1000, and a lower δ. Since the seed's δ_N falls with
  depth, the gain is not a depth effect.
* **Numerical vs provable.** The seeds' numerical δ at N = 8000 is slightly *above* the Alladi–Robinson
  theorem's δ_eq (log(3/2): 0.3954 vs 0.3925, log(6/5): 0.5128 vs 0.5109), because the reduced
  denominators are smaller than the proof's. That overshoot (≤ 0.003) is much smaller than the margin of
  the candidates over δ_eq (log(3/2) +0.037, log(4/3) +0.024, log(5/4) +0.018, log(6/5) +0.014).
* **The forms.** For log(1 + 1/k) the best trajectories all have the form
  L = (2 + c₁)/((2k + 1) + (k + 1)·c₁), with start (1, 1, 3) and direction ≈ (k', k' ± 1, 2k').
* **Shape of the best directions.** All of them are (k, k ± 1, 2k), with δ tied exactly across the
  symmetric partners (k ± 1 swapped between x₀ and x₁, the same approximant sequence). For log 2 the
  analogous direction is Rukhadze's (7, 6; 14) (paper Table 3, D walk), the step that improved
  Alladi–Robinson's log 2 measure.
* **Search limit.** Several best directions are close to the length limit $\|v\|_1 \le 100$ (73–89), so
  longer directions might score higher.
* **From numbers to a theorem.** A record needs the exact denominators' arithmetic (a divisibility
  lemma, as in Alladi–Robinson / Rukhadze) and the asymptotic exponent. The numbers here only show where
  to look.
* **Paper Table 3, log(3/2).** The table gives 0.408 for the (2, 2; 4)/(1, 1; 2) seed, but this seed gives
  δ_1000 = 0.4034 (the other seven rows match at N = 1000).